# PyTorch AutoGrad

In [2]:
import torch

## The AutoGrad DAG

In [38]:
def print_graph(fn, indent=0):
    print("  " * indent + str(fn))
    if hasattr(fn, "next_functions"):
        for f, _ in fn.next_functions:
            if f is not None:
                print_graph(f, indent + 1)
    if hasattr(fn, "variable"):
        print("  " * (indent + 1) + str(fn.variable) + f" (grad={fn.variable.grad})")

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

y = W * x
z = y + b

print("z.grad_fn:")            
print_graph(z.grad_fn)

# Basically:
#               y
#               |
#          MulBackward0
#             /    \
# AccumulateGrad  AccumulateGrad
#      |               |
#      x               W

z.grad_fn:
      tensor(3., requires_grad=True) (grad=None)
      tensor(2., requires_grad=True) (grad=None)
    tensor(4., requires_grad=True) (grad=None)


## The backward() method

In [41]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

y = W * x # y = 6
z = y + b # z = 10

z.backward()

print("z.grad_fn:")              
print_graph(z.grad_fn)     # Now grad values are set!

# The gradient of b is simply dz/db = 1
# The gradient of y is simply dz/dy = 1
# The gradient of x (w.r.t. z!) is simply dz/dx = dz/dy * dy/dx = 1 * W = 3
# The gradient of W (w.r.t. z!) is simply dz/dW = dz/dy * dy/dW = 1 * x = 2 

z.grad_fn:
      tensor(3., requires_grad=True) (grad=2.0)
      tensor(2., requires_grad=True) (grad=3.0)
    tensor(4., requires_grad=True) (grad=1.0)


## Multi-Dimensional

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
W = torch.tensor([3.0, 4.0, 5.0], requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

ground_truth = torch.tensor([8.0, 9.0, 12.0], requires_grad=False)

y = W * x # y = 6
z = y + b # z = 10

loss = torch.nn.functional.mse_loss(z, ground_truth)

loss.backward()


# I have for example the gradient of b w.r.t. the loss
print(f"b.grad {b.grad}")
# And of x and W
print(f"x.grad {x.grad}")
print(f"W.grad {W.grad}")
# Notice how only leaf tensors ("AccumulateGrad") have a grad value. 
# All the others only have grad_fn (they have a gradient, it's just not "accumulated" cause we do not needed for optimizer.step())

b.grad 4.6666669845581055
x.grad 18.666667938232422
W.grad tensor([2.6667, 4.0000, 2.6667])
